# CUDA llama-cpp-python Setup for Vast AI

Run this first inside a Vast AI Jupyter notebook. It checks the GPU, installs helper packages, reinstalls a CUDA `llama-cpp-python` wheel, and verifies GPU offload support.


In [ ]:
import importlib.util
import re
import subprocess
import sys

# CUDA wheel setting:
# - 'auto' chooses cu130 when nvidia-smi reports CUDA 13.x, otherwise cu125.
# - For CUDA 12.x images, cu125 is usually the right wheel.
# - If needed, set explicitly: 'cu124', 'cu125', 'cu130', etc.
LLAMA_CPP_CUDA_WHEEL = 'auto'
REINSTALL_LLAMA_CPP = True

HELPER_PACKAGES = {
    'openpyxl': 'openpyxl',
    'requests': 'requests',
    'tqdm': 'tqdm',
    'huggingface_hub': 'huggingface_hub',
}


def package_available(import_name):
    return importlib.util.find_spec(import_name) is not None


def pip_install(args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *args])


def detect_cuda_wheel():
    if LLAMA_CPP_CUDA_WHEEL != 'auto':
        return LLAMA_CPP_CUDA_WHEEL
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
        text = (result.stdout or '') + (result.stderr or '')
        match = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', text)
        if match and int(match.group(1)) >= 13:
            return 'cu130'
    except FileNotFoundError:
        pass
    return 'cu125'


print('Notebook Python executable:', sys.executable)
print()
print('GPU status:')
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found on PATH.')

print()
print('CUDA compiler version, if available:')
try:
    subprocess.run(['nvcc', '--version'], check=False)
except FileNotFoundError:
    print('nvcc not found on PATH. This is okay when using a prebuilt CUDA wheel.')

missing_helpers = [pip_name for import_name, pip_name in HELPER_PACKAGES.items() if not package_available(import_name)]
if missing_helpers:
    print('Installing missing helper package(s):', ', '.join(missing_helpers))
    pip_install(missing_helpers)
else:
    print('Helper packages are already installed.')

cuda_wheel = detect_cuda_wheel()
extra_index_url = f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_wheel}'
print('llama-cpp-python CUDA wheel:', cuda_wheel)
print('Wheel index:', extra_index_url)

if REINSTALL_LLAMA_CPP or not package_available('llama_cpp'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'llama-cpp-python'], check=False)
    pip_install(['--upgrade', '--force-reinstall', '--no-cache-dir', 'llama-cpp-python', '--extra-index-url', extra_index_url])
else:
    print('llama-cpp-python already installed and REINSTALL_LLAMA_CPP is False.')

from llama_cpp import llama_cpp
print('GPU offload support:', llama_cpp.llama_supports_gpu_offload())


In [ ]:
import importlib.util
import subprocess
import sys

print('Notebook Python executable:', sys.executable)
print('llama_cpp import spec found:', importlib.util.find_spec('llama_cpp') is not None)
print()
print('GPU status:')
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found on PATH.')
print()
print('Package info:')
subprocess.run([sys.executable, '-m', 'pip', 'show', 'llama-cpp-python'], check=False)


## Download Model

Run this after the CUDA setup cell. It downloads the GGUF model into the local Hugging Face cache and sets `GGUF_MODEL_PATH` for the rest of the notebook.


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO_ID = 'unsloth/gemma-4-26B-A4B-it-qat-GGUF'
MODEL_FILENAME = 'gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf'
import os
HF_TOKEN = os.getenv('HF_TOKEN', '')  # Optional. Use only if the model is gated/private.

model_path = hf_hub_download(
    repo_id=MODEL_REPO_ID,
    filename=MODEL_FILENAME,
    token=HF_TOKEN or None,
)

print(f'Model downloaded to: {model_path}')
GGUF_MODEL_PATH = model_path


## 0. Variables

Edit this one cell for the input path, API keys, model path, and llama settings. Output filenames are generated automatically beside `INPUT_PATH`.


In [ ]:
# Edit environment variables or this cell first, then run the helper/action cells below.
import os

# File path. Put the exact path to your .csv or .xlsx file here.
INPUT_PATH = os.getenv('SPREADSHEET_LLM_INPUT_PATH', '/workspace/all_branch_data.csv')  # Example: '/workspace/all_branch_data.csv' or '/workspace/branch_united.xlsx'
SHEET_NAME = 'branch'  # Used only for .xlsx files.
PROCESS_ROW_LIMIT = 50  # Example: 10 for testing, None for all rows after a successful test run.

# Fixed required column names. These are case-sensitive.
LONGITUDE_COLUMN = 'longitude'
LATITUDE_COLUMN = 'latitude'
FULL_ADDRESS_COLUMN = 'full_address'

# Google Geocoding settings. Required only for coordinate filling or run-both.
GOOGLE_GEOCODING_API_KEY = os.getenv('GOOGLE_GEOCODING_API_KEY', '')

# Local GGUF parsing settings. Required only for address parsing or run-both.
# The Download Model cell sets GGUF_MODEL_PATH automatically. You can also paste a .gguf path here.
GGUF_MODEL_PATH = os.getenv('GGUF_MODEL_PATH', globals().get('GGUF_MODEL_PATH', ''))
N_CTX = 2048
N_GPU_LAYERS = -1  # -1 means offload all possible layers to GPU. Use 0 for CPU-only troubleshooting.
N_BATCH = 64
N_UBATCH = 64
FLASH_ATTN = False
MAX_TOKENS = 180
VERBOSE_LLAMA = False

print('Input path:', INPUT_PATH)
print('Variables loaded. Next run Shared Helper Functions, then choose a run cell.')


## 0.1 Shared Helper Functions

Run once after the variables cell. This loads file, geocoding, and address parsing helpers.

In [ ]:
import csv, json, math, os, re, subprocess, sys, time
from pathlib import Path


def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

try:
    import requests
    from openpyxl import Workbook, load_workbook
    from tqdm.auto import tqdm
except ImportError:
    pip_install(['requests', 'openpyxl', 'tqdm'])
    import requests
    from openpyxl import Workbook, load_workbook
    from tqdm.auto import tqdm

NULL_LIKE_VALUES = {'', 'none', 'null', 'nan', 'n/a', 'na', '-', '--'}
PARSED_ADDRESS_COLUMNS = ['barangay2', 'city_municipality2', 'province2', 'region2', 'zip_code2', 'country2', 'island_group2']
ADDRESS_FIELDS = ['barangay', 'city_municipality', 'province', 'region', 'zip_code', 'country', 'island_group']
LLM = globals().get('LLM', None)


def normalize_spaces(value):
    return re.sub(r'\s+', ' ', str(value or '')).strip()


def is_blank_like(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return normalize_spaces(value).lower() in NULL_LIKE_VALUES


def parse_coordinate(value, kind):
    if is_blank_like(value) or isinstance(value, bool):
        return None
    try:
        number = float(str(value).strip())
    except (TypeError, ValueError):
        return None
    if math.isnan(number) or math.isinf(number):
        return None
    if kind == LATITUDE_COLUMN and not -90 <= number <= 90:
        return None
    if kind == LONGITUDE_COLUMN and not -180 <= number <= 180:
        return None
    return number


def header_map(sheet):
    return {normalize_spaces(cell.value): cell.column for cell in sheet[1] if normalize_spaces(cell.value)}


def require_columns(sheet, names):
    headers = header_map(sheet)
    missing = [name for name in names if name not in headers]
    if missing:
        raise ValueError(f'Missing required exact column name(s): {missing}. Found: {list(headers)}')
    return headers


def last_header_column(sheet):
    return max([cell.column for cell in sheet[1] if normalize_spaces(cell.value)] or [0])


def append_fresh_columns(sheet, names):
    next_col = last_header_column(sheet) + 1
    output_cols = {}
    for name in names:
        sheet.cell(row=1, column=next_col).value = name
        output_cols[name] = next_col
        next_col += 1
    return output_cols


def load_file(path, sheet_name):
    if path.suffix.lower() == '.csv':
        workbook = Workbook()
        sheet = workbook.active
        sheet.title = path.stem[:31] or 'csv'
        with path.open('r', newline='', encoding='utf-8-sig') as handle:
            for row in csv.reader(handle):
                sheet.append(row)
        return workbook, sheet, 'csv'
    if path.suffix.lower() == '.xlsx':
        workbook = load_workbook(path)
        if sheet_name not in workbook.sheetnames:
            raise ValueError(f'Workbook does not contain sheet {sheet_name!r}. Available: {workbook.sheetnames}')
        return workbook, workbook[sheet_name], 'xlsx'
    raise ValueError('INPUT_PATH must end in .xlsx or .csv')


def save_file(workbook, sheet, kind, output_path):
    if kind == 'csv':
        with output_path.open('w', newline='', encoding='utf-8') as handle:
            writer = csv.writer(handle)
            for row in sheet.iter_rows(values_only=True):
                writer.writerow(['' if value is None else value for value in row])
    else:
        workbook.save(output_path)


def load_configured_file(input_path=None):
    path = Path(input_path or INPUT_PATH)
    if not str(path) or not path.exists():
        raise ValueError('Set INPUT_PATH to an existing .xlsx or .csv file path in the Variables cell.')
    return load_file(path, SHEET_NAME)


def output_path_for_suffix(input_path, suffix):
    path = Path(input_path)
    if not str(path):
        raise ValueError('Set INPUT_PATH in the Variables cell.')
    if not suffix.startswith('_'):
        raise ValueError('Output suffix must start with an underscore, such as _coords, _parsed, or _cleaned.')
    return path.with_name(f'{path.stem}{suffix}{path.suffix}')


def save_configured_file(workbook, sheet, kind, output_suffix):
    path = output_path_for_suffix(INPUT_PATH, output_suffix)
    save_file(workbook, sheet, kind, path)
    return path


def geocode(address, api_key):
    response = requests.get('https://maps.googleapis.com/maps/api/geocode/json', params={'address': address, 'key': api_key}, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if payload.get('status') != 'OK' or not payload.get('results'):
        return None
    location = payload['results'][0].get('geometry', {}).get('location', {})
    latitude = parse_coordinate(location.get('lat'), LATITUDE_COLUMN)
    longitude = parse_coordinate(location.get('lng'), LONGITUDE_COLUMN)
    return None if latitude is None or longitude is None else (latitude, longitude)


def add_missing_coordinates(workbook, sheet, kind, api_key=None):
    api_key = api_key or GOOGLE_GEOCODING_API_KEY
    if not api_key:
        raise ValueError('Set GOOGLE_GEOCODING_API_KEY in the Variables cell.')
    headers = require_columns(sheet, [LONGITUDE_COLUMN, LATITUDE_COLUMN, FULL_ADDRESS_COLUMN])
    longitude_col = headers[LONGITUDE_COLUMN]
    latitude_col = headers[LATITUDE_COLUMN]
    address_col = headers[FULL_ADDRESS_COLUMN]
    last_row = sheet.max_row if PROCESS_ROW_LIMIT is None else min(sheet.max_row, PROCESS_ROW_LIMIT + 1)
    summary = {'rows_checked': max(last_row - 1, 0), 'rows_needing_coordinates': 0, 'rows_geocoded': 0, 'skipped_blank_address_rows': [], 'no_result_rows': [], 'failed_rows': {}}

    for row_number in tqdm(range(2, last_row + 1), desc='Adding missing coordinates'):
        latitude_missing = parse_coordinate(sheet.cell(row=row_number, column=latitude_col).value, LATITUDE_COLUMN) is None
        longitude_missing = parse_coordinate(sheet.cell(row=row_number, column=longitude_col).value, LONGITUDE_COLUMN) is None
        if not (latitude_missing or longitude_missing):
            continue
        summary['rows_needing_coordinates'] += 1
        address = normalize_spaces(sheet.cell(row=row_number, column=address_col).value)
        if not address:
            summary['skipped_blank_address_rows'].append(row_number)
            continue
        try:
            result = geocode(address, api_key)
        except Exception as exc:
            summary['failed_rows'][row_number] = str(exc)
            continue
        if result is None:
            summary['no_result_rows'].append(row_number)
            continue
        latitude, longitude = result
        sheet.cell(row=row_number, column=latitude_col).value = latitude
        sheet.cell(row=row_number, column=longitude_col).value = longitude
        summary['rows_geocoded'] += 1
        time.sleep(0.05)
    summary['failed_row_count'] = len(summary.pop('failed_rows'))
    return summary


def extract_json_object_text(text):
    stripped = str(text or '').strip()
    if stripped.startswith('```'):
        stripped = re.sub(r'^```(?:json)?\s*', '', stripped, flags=re.I)
        stripped = re.sub(r'\s*```$', '', stripped)
    start, end = stripped.find('{'), stripped.rfind('}')
    if start < 0 or end < start:
        raise ValueError('No JSON object found in model response.')
    return stripped[start:end + 1]


def normalize_island_group(value):
    text = normalize_spaces(value).lower()
    if is_blank_like(text):
        return None
    if 'luzon' in text:
        return 'Luzon'
    if 'visayas' in text or 'visaya' in text:
        return 'Visayas'
    if 'mindanao' in text:
        return 'Mindanao'
    return None


def normalize_address_result(raw):
    if isinstance(raw, str):
        raw = json.loads(extract_json_object_text(raw))
    if not isinstance(raw, dict):
        raise ValueError('Model returned a non-object JSON value.')
    aliases = {'barangay': 'barangay', 'city_municipality': 'city_municipality', 'city/municipality': 'city_municipality', 'city': 'city_municipality', 'municipality': 'city_municipality', 'province': 'province', 'region': 'region', 'zip_code': 'zip_code', 'zip code': 'zip_code', 'zipcode': 'zip_code', 'postal_code': 'zip_code', 'country': 'country', 'island_group': 'island_group', 'island group': 'island_group', 'islandgroup': 'island_group'}
    normalized = {field: None for field in ADDRESS_FIELDS}
    for key, value in raw.items():
        canonical = aliases.get(normalize_spaces(key).lower())
        if canonical not in normalized:
            continue
        text = normalize_spaces(value)
        normalized[canonical] = normalize_island_group(text) if canonical == 'island_group' else (None if is_blank_like(text) else text)
    return normalized


def build_prompt(address):
    return f'''You parse Philippine addresses into administrative location fields.
Return only JSON with keys: barangay, city_municipality, province, region, zip_code, country, island_group.
Rules:
- Building names, company names, malls, roads, streets, drives, highways, subdivisions, landmarks, and branch names are not barangays, cities, or provinces.
- If barangay is not explicitly written as Brgy/Barangay or clearly known, use null.
- Never copy city_municipality into barangay unless the address explicitly marks that same value as Brgy/Barangay.
- ZIP code must be null unless a numeric postal code appears in the address.
- For Metro Manila cities, province is "Metro Manila" and region is "NCR".
- island_group must be exactly "Luzon", "Visayas", or "Mindanao"; use null if not confidently known.
- Region (For region, return the official Philippine region code format only, such as "Region I", "Region II", "Region III", "Region IV-A", "Region VI", etc. Do not return the region name unless separately requested.)
- Country should be "Philippines" when the address is in the Philippines.
Example:
Address: Ropali Plaza, Escriva Drive, Pasig, Metro Manila
{{"barangay":null,"city_municipality":"Pasig","province":"Metro Manila","region":"NCR","zip_code":null,"country":"Philippines","island_group":"Luzon"}}
Now parse this address:
{address}'''


def build_repair_prompt(address, previous_output):
    return f'''The previous response was not valid JSON. Return only one valid JSON object with exactly these keys: barangay, city_municipality, province, region, zip_code, country, island_group. Use null for missing values. island_group must be Luzon, Visayas, or Mindanao when known.
Address:
{address}
Invalid previous response:
{previous_output}'''


def model_generate(llm, prompt):
    messages = [{'role': 'system', 'content': 'You are a strict JSON extraction engine. Return only one compact JSON object. No markdown.'}, {'role': 'user', 'content': prompt}]
    output = llm.create_chat_completion(messages=messages, max_tokens=MAX_TOKENS, temperature=0)
    choice = output['choices'][0]
    message = choice.get('message') or {}
    return message.get('content') or choice.get('text') or ''


def parse_address(llm, address):
    first_output = model_generate(llm, build_prompt(address))
    try:
        return normalize_address_result(first_output)
    except Exception:
        return normalize_address_result(model_generate(llm, build_repair_prompt(address, first_output)))


def write_parsed_row(sheet, row_number, output_columns, parsed):
    for field, column_name in zip(ADDRESS_FIELDS, PARSED_ADDRESS_COLUMNS, strict=True):
        sheet.cell(row=row_number, column=output_columns[column_name]).value = parsed.get(field)



def load_llm_if_needed(llm=None):
    llm = llm or globals().get('LLM')
    if llm is not None:
        return llm
    from llama_cpp import Llama
    model_path = Path(GGUF_MODEL_PATH)
    if not GGUF_MODEL_PATH or not model_path.exists():
        raise ValueError('Set GGUF_MODEL_PATH to an existing .gguf model file path. Run the Download Model cell or paste the path in Variables.')
    print('Loading local model:', model_path)
    print(f'Model settings: N_CTX={N_CTX}, N_GPU_LAYERS={N_GPU_LAYERS}, N_BATCH={N_BATCH}, N_UBATCH={N_UBATCH}, FLASH_ATTN={FLASH_ATTN}')
    try:
        loaded = Llama(
            model_path=str(model_path),
            n_ctx=N_CTX,
            n_gpu_layers=N_GPU_LAYERS,
            n_batch=N_BATCH,
            n_ubatch=N_UBATCH,
            flash_attn=FLASH_ATTN,
            verbose=VERBOSE_LLAMA,
        )
    except ValueError as exc:
        if 'llama_context' in str(exc):
            raise ValueError(
                'Failed to create llama_context. This is usually a memory or model-loading issue. '
                'Try these settings in the Variables cell, then rerun from Shared Helper Functions onward: '
                'N_CTX = 1024, N_BATCH = 64, N_UBATCH = 64, FLASH_ATTN = False. '
                'If it still fails, set N_GPU_LAYERS = 0 for CPU-only loading, or use a smaller/stronger-quantized GGUF model.'
            ) from exc
        raise
    globals()['LLM'] = loaded
    return loaded


def parse_full_address(workbook, sheet, kind, llm=None):
    llm = llm or globals().get('LLM')
    if llm is None:
        raise ValueError('LLM is not loaded. Run load_llm_if_needed() or pass an llm object.')
    headers = require_columns(sheet, [FULL_ADDRESS_COLUMN])
    address_col = headers[FULL_ADDRESS_COLUMN]
    output_columns = append_fresh_columns(sheet, PARSED_ADDRESS_COLUMNS)
    last_row = sheet.max_row if PROCESS_ROW_LIMIT is None else min(sheet.max_row, PROCESS_ROW_LIMIT + 1)
    summary = {'rows_checked': max(last_row - 1, 0), 'rows_parsed': 0, 'skipped_blank_address_rows': [], 'failed_rows': {}, 'appended_columns': PARSED_ADDRESS_COLUMNS}
    for row_number in tqdm(range(2, last_row + 1), desc='Parsing full address'):
        address = normalize_spaces(sheet.cell(row=row_number, column=address_col).value)
        if not address:
            summary['skipped_blank_address_rows'].append(row_number)
            write_parsed_row(sheet, row_number, output_columns, {field: None for field in ADDRESS_FIELDS})
            continue
        try:
            parsed = parse_address(llm, address)
            summary['rows_parsed'] += 1
        except Exception as exc:
            summary['failed_rows'][row_number] = str(exc)
            parsed = {field: None for field in ADDRESS_FIELDS}
        write_parsed_row(sheet, row_number, output_columns, parsed)
    summary['failed_row_count'] = len(summary.pop('failed_rows'))
    return summary

print('Shared helper functions loaded for Vast AI / CUDA llama-cpp-python backend.')


## 1. Add Missing Longitude / Latitude

Uses `full_address` exactly as the Google Geocoding query and fills rows where `longitude` or `latitude` is blank, null-like, or invalid. Existing rows with both valid coordinates are not overwritten.


In [ ]:
workbook, sheet, kind = load_configured_file()
summary = add_missing_coordinates(workbook, sheet, kind)
saved_path = save_configured_file(workbook, sheet, kind, '_coords')
print('Saved:', saved_path)
print(json.dumps(summary, indent=2))


## 2. Parse Full Address

Uses `full_address`, loads the local GGUF model if needed, and appends fresh parsed address columns.


In [ ]:
LLM = load_llm_if_needed()
workbook, sheet, kind = load_configured_file()
summary = parse_full_address(workbook, sheet, kind, llm=LLM)
saved_path = save_configured_file(workbook, sheet, kind, '_parsed')
print('Saved:', saved_path)
print(json.dumps(summary, indent=2))


## 3. Run Both

Adds missing coordinates first, immediately saves a `_coords` checkpoint, then parses full addresses and saves one `_cleaned` output file if parsing succeeds.

In [ ]:
workbook, sheet, kind = load_configured_file()
coordinate_summary = add_missing_coordinates(workbook, sheet, kind)
coords_saved_path = save_configured_file(workbook, sheet, kind, '_coords')
print('Saved coordinate checkpoint:', coords_saved_path)

LLM = load_llm_if_needed()
parse_summary = parse_full_address(workbook, sheet, kind, llm=LLM)
saved_path = save_configured_file(workbook, sheet, kind, '_cleaned')
print('Saved:', saved_path)
print(json.dumps({'coordinates': coordinate_summary, 'full_address_parsing': parse_summary}, indent=2))
